# Mutual Information in Correlated Gaussian

Chapter 14: Information Theory · *Mathematics for AI and Machine Learning* by Fuheng Wu

See mutual information increase with correlation strength.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-14/mutual-info-gaussian/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch
from torch.distributions import MultivariateNormal, Normal

torch.manual_seed(0)

# The demo's default correlation between X and Y (both standard normal)
rho = 0.4
cov = torch.tensor([[1.0, rho], [rho, 1.0]], dtype=torch.float64)
joint = MultivariateNormal(torch.zeros(2, dtype=torch.float64), cov)
marg = Normal(torch.tensor(0.0, dtype=torch.float64), torch.tensor(1.0, dtype=torch.float64))

# ── Closed form: I(X; Y) = -1/2 log(1 - rho^2) ──────────────────────────────
mi_exact = -0.5 * math.log(1 - rho ** 2)
print(f"I(X;Y) = -1/2 log(1 - rho^2) = {mi_exact:.6f} nats")

# Same from entropies: I = H(X) + H(Y) - H(X, Y)
H = lambda d: d.entropy().item()
print("H(X) + H(Y) - H(X,Y) =", 2 * H(marg) - H(joint))

# ── Monte Carlo: I = E[log p(x, y) - log p(x) - log p(y)] ───────────────────
xy = joint.sample((500000,))
mc = (joint.log_prob(xy) - marg.log_prob(xy[:, 0]) - marg.log_prob(xy[:, 1])).mean()
print("Monte Carlo estimate  =", mc.item())

# ── MI grows without bound as |rho| -> 1, and is 0 only at rho = 0 ──────────
print("\n  rho     I(X;Y)")
for r in (0.0, 0.2, 0.4, 0.7, 0.9, 0.99, -0.9):
    print(f"{r:+5.2f}   {-0.5 * math.log(1 - r * r):.4f}")

I(X;Y) = -1/2 log(1 - rho^2) = 0.087177 nats
H(X) + H(Y) - H(X,Y) = 0.08717669357238877
Monte Carlo estimate  = 0.08733691190402056

  rho     I(X;Y)
+0.00   -0.0000
+0.20   0.0204
+0.40   0.0872
+0.70   0.3367
+0.90   0.8304
+0.99   1.9585
-0.90   0.8304
